# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR²: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values.

In [ ]:
# List all available record sets and their fields, using their @id
print("Available Record Sets (by @id):\n")
for rs in metadata.record_sets:
    print(f"- {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            print(f"    - Field: {field['@id']} (dataType: {field.get('dataType', 'unknown')}, source: {field.get('source', {}).get('@id', '-')})")
    if 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        for column in columns:
            print(f"    - Column: {column['@id']} (dataType: {column.get('dataType', 'unknown')}, source: {column.get('source', {}).get('@id', '-')})")

Below we preview the first few records for each record set. *Each entity is referenced by its `@id` as required.*

In [ ]:
# Preview the first 3 records for each record set using their @id
for rs in metadata.record_sets:
    rs_id = rs['@id']
    print(f"\nRecord Set: {rs_id}")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id values into a list for convenience
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print("Record set @ids:", record_set_ids)

# Load all record sets as dataframes (by @id)
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecord set '{rs_id}' loaded. Columns: {list(df.columns)} Total records: {len(df)}")

# Choose the primary record set for demonstration (e.g., the first one)
main_record_set_id = record_set_ids[0]
print(f"\nUsing main record set: {main_record_set_id}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply standard data processing steps such as filtering records, normalizing numeric fields, and grouping by key attributes. All fields are referenced by their `@id` values as required.

In [ ]:
#--- Identify suitable numeric and categorical field @id values for demonstration ---#
main_df = dataframes[main_record_set_id]

# Attempt to infer some numeric and group-column @id fields
print('Sample columns:', list(main_df.columns))

# You may need to replace these with actual @id field names for true data - here, guess typical croissant-named fields
# E.g.: '/record_sets/proteins/fields/mw' for molecular weight
candidate_numeric_fields = [col for col in main_df.columns if 'abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'peptide' in col.lower() or 'number' in col.lower()]
if len(candidate_numeric_fields) == 0:
    # fallback to any float/integer looking column
    candidate_numeric_fields = list(main_df.select_dtypes(include=['float', 'int']).columns)

# Pick the first found numeric field
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
else:
    numeric_field_id = main_df.columns[0]  # fallback to first
print(f"Using numeric field: {numeric_field_id}")

# Find a possible group field (categorical)
candidate_group_fields = [col for col in main_df.columns if 'desc' in col.lower() or 'type' in col.lower() or 'group' in col.lower() or 'class' in col.lower()]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id:
    print(f"Using group field: {group_field_id}")
else:
    print("No group/categorical field detected, will skip grouping.")

threshold = main_df[numeric_field_id].astype(float).mean() if numeric_field_id else 0

# Filter: Only records where numeric_field > mean
filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the group_field if available
if group_field_id and group_field_id in main_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (mean of numeric columns):")
    display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the main numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group if available
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to load, inspect, process, and visualize mass spectrometry dataset records using field and record set `@id` references, following the [FAIR² Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

- All data entities and operations were referenced using their Croissant `@id` fields.
- We showed how to list metadata, extract and convert record sets to DataFrames, filter and normalize quantitative measures, and group or chart the data.
- For further analysis, repeat similar steps, always referencing Croissant schema-defined `@id` identifiers to ensure reproducibility and clarity.

**Explore the Croissant schema documentation for more advanced usages!**